# Task 2 – Preprocessing, Features and Pipeline

Run all cells from top to bottom. This notebook includes its own imports, synthetic dataset, and train/validation/test split setup.

# Week 2 – Weekly Assessment: Machine Learning
**YOHO Technologies | Loan Applicants Scenario**

This notebook completes Tasks 1–4 using one reproducible synthetic bank-loan dataset. Run cells from top to bottom. The test set is kept aside until the final evaluation in Task 4.

In [1]:
import warnings, os, joblib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_curve, roc_auc_score,
                             mean_absolute_error, mean_squared_error, r2_score,
                             silhouette_score)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.cluster import KMeans

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Reproducible stratified cross-validation splitter shared by tasks.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


In [3]:
# Compatibility helper for scikit-learn versions before/after sparse_output rename.
def make_onehot():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


## 0. Dataset Generation – Bank Loan Applicants

In [4]:
n_samples = 2000
rng = np.random.default_rng(RANDOM_STATE)

applicant_ids = np.arange(1001, 1001 + n_samples)
dates = pd.date_range("2024-01-01", periods=n_samples, freq="4h")
age = rng.integers(21, 66, n_samples)
annual_income = rng.lognormal(mean=10.8, sigma=0.6, size=n_samples)
loan_amount = np.clip(annual_income * rng.uniform(0.15, 0.65, n_samples) + rng.normal(0, 5000, n_samples), 5000, 150000)
credit_score = np.clip(rng.normal(670, 60, n_samples), 300, 850)
debt_to_income = np.clip((loan_amount * 0.12) / annual_income + rng.normal(0.15, 0.05, n_samples), 0.01, 0.95)
education = rng.choice(["High School", "Bachelor", "Master", "PhD"], n_samples, p=[0.30, 0.45, 0.20, 0.05])
employment = rng.choice(["Salaried", "Self-Employed", "Business", "Unemployed"], n_samples, p=[0.55, 0.25, 0.15, 0.05])
property_area = rng.choice(["Urban", "Semiurban", "Rural"], n_samples, p=[0.40, 0.35, 0.25])

# A synthetic probability of default, used only to create a realistic target.
logit = (-2.2 + 3.0 * debt_to_income - 0.006 * (credit_score - 650)
         + 0.45 * (employment == "Unemployed")
         + 0.25 * (employment == "Self-Employed")
         + 0.18 * (education == "High School"))
prob_default = 1 / (1 + np.exp(-logit))
default = rng.binomial(1, prob_default)

df = pd.DataFrame({
    "Applicant_ID": applicant_ids,
    "Application_Date": dates,
    "Age": age,
    "Annual_Income": annual_income,
    "Loan_Amount": loan_amount,
    "Credit_Score": credit_score,
    "Debt_to_Income": debt_to_income,
    "Education_Level": education,
    "Employment_Type": employment,
    "Property_Area": property_area,
    "Default": default
})

# Add missing values to demonstrate imputation.
for col in ["Annual_Income", "Credit_Score", "Employment_Type"]:
    missing_idx = rng.choice(df.index, size=int(0.03 * n_samples), replace=False)
    df.loc[missing_idx, col] = np.nan

print("Dataset shape:", df.shape)
print("Default rate:", round(df["Default"].mean(), 3))
display(df.head())

Dataset shape: (2000, 11)
Default rate: 0.18


,Applicant_ID,Application_Date,Age,Annual_Income,Loan_Amount,Credit_Score,Debt_to_Income,Education_Level,Employment_Type,Property_Area,Default
0,1001,2024-01-01 00:00:00,25,31403.817584,21781.751722,572.221835,0.286323,PhD,Salaried,Rural,0
1,1002,2024-01-01 04:00:00,55,85358.443759,39681.941286,619.451077,0.197838,Bachelor,Salaried,Urban,0
2,1003,2024-01-01 08:00:00,50,50049.466458,5000.000000,655.786146,0.190497,Bachelor,Salaried,Rural,0
3,1004,2024-01-01 12:00:00,40,41370.467114,9761.351215,644.974706,0.205203,Bachelor,Salaried,Urban,0
4,1005,2024-01-01 16:00:00,40,45995.131901,22921.204864,650.034598,0.199029,Bachelor,Salaried,Urban,0


## Shared feature engineering and train/validation/test split

In [5]:
def add_features(data):
    data = data.copy()
    dates = pd.to_datetime(data["Application_Date"])
    data["Application_Month"] = dates.dt.month
    data["Application_Day"] = dates.dt.day
    data["Income_to_Loan"] = data["Annual_Income"] / data["Loan_Amount"].replace(0, np.nan)
    data["Log_Annual_Income"] = np.log1p(data["Annual_Income"].clip(lower=0))
    return data

df_features = add_features(df)
target = "Default"
drop_cols = ["Applicant_ID", "Application_Date", target]
X = df_features.drop(columns=drop_cols)
y = df_features[target].astype(int)

# First split: 60% train+validation, 40% test; then split remaining into 60/20 overall.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=RANDOM_STATE, stratify=y_train_val
)
print(f"Train: {len(X_train)} ({len(X_train)/len(X):.0%})")
print(f"Validation: {len(X_val)} ({len(X_val)/len(X):.0%})")
print(f"Test: {len(X_test)} ({len(X_test)/len(X):.0%}) — kept aside until Task 4")

Train: 1200 (60%)
Validation: 400 (20%)
Test: 400 (20%) — kept aside until Task 4


In [6]:
# Shared column lists used by all task notebooks.
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()


## Task 2 – Preprocessing, Features and Pipeline

In [7]:
# Try KNNImputer on numeric columns as requested.
knn_imputer = KNNImputer(n_neighbors=5)
knn_numeric = knn_imputer.fit_transform(X_train[numeric_features])
print("KNNImputer numeric output shape:", knn_numeric.shape)

# Main reusable preprocessing pipeline: median numeric, most-frequent categorical,
# ordinal encoding for education, one-hot encoding for remaining categorical columns.
ordinal_feature = ["Education_Level"]
nominal_features = [c for c in categorical_features if c not in ordinal_feature]
ordinal_categories = [["High School", "Bachelor", "Master", "PhD"]]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), numeric_features),
    ("ord", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("encoder", OrdinalEncoder(categories=ordinal_categories,
                                                 handle_unknown="use_encoded_value",
                                                 unknown_value=-1))]), ordinal_feature),
    ("nom", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("encoder", make_onehot())]), nominal_features)
])

pipeline_task2 = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])
task2_cv = cross_val_score(pipeline_task2, X_train_val, y_train_val, cv=cv, scoring="accuracy")
print(f"Pipeline CV accuracy: {task2_cv.mean():.3f} +/- {task2_cv.std():.3f}")

pipeline_task2.fit(X_train_val, y_train_val)
joblib.dump(pipeline_task2, "loan_default_pipeline.joblib")
loaded_pipeline = joblib.load("loan_default_pipeline.joblib")

new_rows = X_train_val.head(3).copy()
predictions = loaded_pipeline.predict(new_rows)
print("Predictions for 3 new raw rows:", predictions.tolist())

KNNImputer numeric output shape: (1200, 9)


Pipeline CV accuracy: 0.822 +/- 0.002


Predictions for 3 new raw rows: [0, 0, 0]
